In [7]:
from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from typing import Optional

import pandas as pd
from rdflib import Graph, Literal, OWL, RDF, RDFS, URIRef
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# RDF predicates
# ============================================================

DUL_HAS_DATA_VALUE = URIRef(
    "http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#hasDataValue"
)


# ============================================================
# Numeric constraint representation
# ============================================================

@dataclass(frozen=True)
class NumericConstraint:
    value: Decimal
    quantity_type: str
    operator: str
    context: frozenset[str]


# ============================================================
# URI normalization
# ============================================================

def local_name(value) -> str:
    """
    Convert an RDF URI into a readable local name.

    Examples:
        fred:fabric_1 -> fabric
        fred:percentage-entity_1 -> percentage-entity
        pbrs:weave-01 -> weave-01
    """
    if isinstance(value, Literal):
        return str(value).strip().lower()

    text = str(value).rstrip("/")

    if "#" in text:
        name = text.rsplit("#", 1)[-1]
    else:
        name = text.rsplit("/", 1)[-1]

    # Remove generated instance suffixes without regex:
    # fabric_1 -> fabric
    # percentage-entity_23 -> percentage-entity
    parts = name.rsplit("_", 1)

    if len(parts) == 2 and parts[1].isdigit():
        name = parts[0]

    return name.lower().strip()


# ============================================================
# Graph loading
# ============================================================

def load_turtle(turtle_text: str) -> Graph:
    graph = Graph()
    graph.parse(data=turtle_text, format="turtle")
    return graph


# ============================================================
# Schema filtering
# ============================================================

def is_schema_triple(subject, predicate, obj) -> bool:
    """
    Ignore ontology/schema declarations because they occur in many
    AMR2FRED graphs and would artificially inflate similarity.
    """
    if predicate in {
        RDFS.label,
        RDFS.subClassOf,
        RDFS.domain,
        RDFS.range,
    }:
        return True

    if predicate == RDF.type and obj in {
        OWL.ObjectProperty,
        OWL.DatatypeProperty,
        OWL.Class,
        OWL.AnnotationProperty,
    }:
        return True

    return False


# ============================================================
# Node descriptions
# ============================================================

def node_types(graph: Graph, node) -> set[str]:
    """
    Return meaningful RDF types assigned to an instance node.
    """
    results = set()

    for obj in graph.objects(node, RDF.type):
        if obj in {
            OWL.ObjectProperty,
            OWL.DatatypeProperty,
            OWL.Class,
            OWL.AnnotationProperty,
        }:
            continue

        results.add(local_name(obj))

    return results


def node_signature(graph: Graph, node) -> str:
    """
    Represent a node by its RDF types rather than generated URI IDs.
    """
    if isinstance(node, Literal):
        datatype = local_name(node.datatype) if node.datatype else "literal"
        return f"{datatype}:{local_name(node)}"

    types = node_types(graph, node)

    if types:
        return "+".join(sorted(types))

    return local_name(node)


# ============================================================
# Structural graph features
# ============================================================

def graph_features(graph: Graph) -> list[str]:
    """
    Automatically create graph features.

    No specific FRED, PropBank or DUL tag names are predefined.
    """
    features: list[str] = []

    instance_triples = [
        (subject, predicate, obj)
        for subject, predicate, obj in graph
        if not is_schema_triple(subject, predicate, obj)
    ]

    # Individual triple features
    for subject, predicate, obj in instance_triples:
        subject_name = node_signature(graph, subject)
        predicate_name = local_name(predicate)
        object_name = node_signature(graph, obj)

        features.append(f"predicate::{predicate_name}")
        features.append(f"subject::{subject_name}")
        features.append(f"object::{object_name}")

        features.append(
            f"triple::{subject_name}"
            f"--{predicate_name}"
            f"--{object_name}"
        )

    # Two-edge paths:
    # A --predicate1--> B --predicate2--> C
    nodes = set()

    for subject, _, obj in instance_triples:
        nodes.add(subject)

        if not isinstance(obj, Literal):
            nodes.add(obj)

    for middle_node in nodes:
        incoming = []

        for source, predicate in graph.subject_predicates(middle_node):
            if is_schema_triple(source, predicate, middle_node):
                continue

            incoming.append((source, predicate))

        outgoing = []

        for predicate, target in graph.predicate_objects(middle_node):
            if is_schema_triple(middle_node, predicate, target):
                continue

            outgoing.append((predicate, target))

        for source, predicate1 in incoming:
            for predicate2, target in outgoing:
                features.append(
                    "path2::"
                    f"{node_signature(graph, source)}"
                    f"--{local_name(predicate1)}"
                    f"--{node_signature(graph, middle_node)}"
                    f"--{local_name(predicate2)}"
                    f"--{node_signature(graph, target)}"
                )

    return features


# ============================================================
# Numeric constraint extraction
# ============================================================

def numeric_context(
    graph: Graph,
    numeric_node,
    depth: int = 2,
) -> frozenset[str]:
    """
    Build a generic neighborhood signature around a numeric node.

    This helps distinguish:
        percentage = 5
    from:
        heading number = 5807
    or:
        weight = 5
    """
    context = set()
    frontier = {numeric_node}
    visited = {numeric_node}

    for level in range(depth):
        next_frontier = set()

        for node in frontier:
            # Incoming relationships
            for subject, predicate in graph.subject_predicates(node):
                if is_schema_triple(subject, predicate, node):
                    continue

                context.add(
                    f"depth{level + 1}:incoming:"
                    f"{local_name(predicate)}:"
                    f"{node_signature(graph, subject)}"
                )

                if subject not in visited:
                    visited.add(subject)
                    next_frontier.add(subject)

            # Outgoing relationships
            for predicate, obj in graph.predicate_objects(node):
                if is_schema_triple(node, predicate, obj):
                    continue

                if predicate == DUL_HAS_DATA_VALUE:
                    continue

                context.add(
                    f"depth{level + 1}:outgoing:"
                    f"{local_name(predicate)}:"
                    f"{node_signature(graph, obj)}"
                )

                if not isinstance(obj, Literal) and obj not in visited:
                    visited.add(obj)
                    next_frontier.add(obj)

        frontier = next_frontier

    return frozenset(context)


def infer_operator(graph: Graph, numeric_node) -> str:
    """
    Infer the comparison operator from nodes connected to the number.

    The exact URI is not predefined. The operator is discovered from
    connected node names and RDF types.
    """
    related_names = set()

    # Search outgoing nodes
    for _, obj in graph.predicate_objects(numeric_node):
        if isinstance(obj, Literal):
            continue

        related_names.add(local_name(obj))
        related_names.update(node_types(graph, obj))

    # Search incoming nodes
    for subject, _ in graph.subject_predicates(numeric_node):
        related_names.add(local_name(subject))
        related_names.update(node_types(graph, subject))

    normalized = {
        name.replace("_", "-").replace(" ", "-")
        for name in related_names
    }

    if any(
        name in {
            "at-least",
            "minimum",
            "not-less-than",
            "no-less-than",
        }
        for name in normalized
    ):
        return ">="

    if any(
        name in {
            "at-most",
            "maximum",
            "not-more-than",
            "no-more-than",
        }
        for name in normalized
    ):
        return "<="

    if any(
        name in {
            "more-than",
            "greater-than",
            "above",
            "over",
        }
        for name in normalized
    ):
        return ">"

    if any(
        name in {
            "less-than",
            "below",
            "under",
        }
        for name in normalized
    ):
        return "<"

    return "="


def extract_numeric_constraints(
    graph: Graph,
) -> list[NumericConstraint]:
    """
    Extract all nodes that have dul:hasDataValue.
    """
    constraints = []

    for numeric_node, _, literal in graph.triples(
        (None, DUL_HAS_DATA_VALUE, None)
    ):
        if not isinstance(literal, Literal):
            continue

        try:
            value = Decimal(str(literal))
        except InvalidOperation:
            continue

        quantity_types = node_types(graph, numeric_node)

        quantity_type = (
            "+".join(sorted(quantity_types))
            if quantity_types
            else local_name(numeric_node)
        )

        constraints.append(
            NumericConstraint(
                value=value,
                quantity_type=quantity_type,
                operator=infer_operator(graph, numeric_node),
                context=numeric_context(graph, numeric_node),
            )
        )

    return constraints


# ============================================================
# Numeric context matching
# ============================================================

def jaccard_similarity(
    set_a: frozenset[str],
    set_b: frozenset[str],
) -> float:
    union = set_a | set_b

    if not union:
        return 1.0

    return len(set_a & set_b) / len(union)


def same_numeric_context(
    constraint_a: NumericConstraint,
    constraint_b: NumericConstraint,
    context_threshold: float = 0.25,
) -> bool:
    """
    Decide whether two numbers refer to the same kind of quantity.
    """
    if constraint_a.quantity_type != constraint_b.quantity_type:
        return False

    return (
        jaccard_similarity(
            constraint_a.context,
            constraint_b.context,
        )
        >= context_threshold
    )


# ============================================================
# Constraint comparison
# ============================================================

def constraints_equivalent(
    constraint_a: NumericConstraint,
    constraint_b: NumericConstraint,
) -> bool:
    """
    Test whether two constraints express the same numeric rule.
    """
    return (
        constraint_a.value == constraint_b.value
        and constraint_a.operator == constraint_b.operator
    )


def candidate_satisfies_requirement(
    candidate: NumericConstraint,
    requirement: NumericConstraint,
) -> bool:
    """
    Determine whether a candidate numeric value satisfies a rule.

    Examples:
        candidate 4% against at least 5% -> False
        candidate 6% against at least 5% -> True
        candidate 8% against exactly 8% -> True
    """
    candidate_value = candidate.value
    required_value = requirement.value

    if requirement.operator == ">=":
        return candidate_value >= required_value

    if requirement.operator == "<=":
        return candidate_value <= required_value

    if requirement.operator == ">":
        return candidate_value > required_value

    if requirement.operator == "<":
        return candidate_value < required_value

    return candidate_value == required_value


def numeric_equivalence_score(
    graph_a: Graph,
    graph_b: Graph,
    context_threshold: float = 0.25,
) -> float:
    """
    Return 1 only when corresponding numeric constraints are equivalent.

    This is strict semantic-equivalence matching.
    """
    constraints_a = extract_numeric_constraints(graph_a)
    constraints_b = extract_numeric_constraints(graph_b)

    if not constraints_a and not constraints_b:
        return 1.0

    if not constraints_a or not constraints_b:
        return 0.0

    matched_b = set()

    for constraint_a in constraints_a:
        found_match = False

        for index_b, constraint_b in enumerate(constraints_b):
            if index_b in matched_b:
                continue

            if not same_numeric_context(
                constraint_a,
                constraint_b,
                context_threshold,
            ):
                continue

            if constraints_equivalent(
                constraint_a,
                constraint_b,
            ):
                matched_b.add(index_b)
                found_match = True
                break

        if not found_match:
            return 0.0

    return 1.0


def requirement_satisfaction_score(
    candidate_graph: Graph,
    requirement_graph: Graph,
    context_threshold: float = 0.25,
) -> float:
    """
    Return 1 if every numeric requirement in requirement_graph is
    satisfied by a corresponding value in candidate_graph.

    Example:
        candidate graph contains 4%
        requirement graph contains at least 5%
        result = 0
    """
    candidate_constraints = extract_numeric_constraints(candidate_graph)
    requirement_constraints = extract_numeric_constraints(
        requirement_graph
    )

    if not requirement_constraints:
        return 1.0

    if not candidate_constraints:
        return 0.0

    for requirement in requirement_constraints:
        satisfied = False

        for candidate in candidate_constraints:
            if not same_numeric_context(
                candidate,
                requirement,
                context_threshold,
            ):
                continue

            if candidate_satisfies_requirement(
                candidate,
                requirement,
            ):
                satisfied = True
                break

        if not satisfied:
            return 0.0

    return 1.0


# ============================================================
# Structural similarity
# ============================================================

def structural_similarity_matrix(
    graphs: list[Graph],
) -> list[list[float]]:
    documents = [
        " ".join(graph_features(graph))
        for graph in graphs
    ]

    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        sublinear_tf=True,
    )

    feature_matrix = vectorizer.fit_transform(documents)

    return cosine_similarity(feature_matrix).tolist()


# ============================================================
# Final all-versus-all comparison
# ============================================================

def compare_all_graphs(
    turtle_graphs: list[str],
    names: Optional[list[str]] = None,
    comparison_mode: str = "equivalence",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    comparison_mode:

        "equivalence"
            Requires the same value and same operator.

            5% != at least 5%
            4% != at least 5%

        "requirement"
            Treat graph j as a requirement and graph i as a candidate.

            4% does not satisfy at least 5%
            6% satisfies at least 5%
    """
    if comparison_mode not in {
        "equivalence",
        "requirement",
    }:
        raise ValueError(
            "comparison_mode must be "
            "'equivalence' or 'requirement'"
        )

    graphs = [
        load_turtle(turtle)
        for turtle in turtle_graphs
    ]

    if names is None:
        names = [
            f"graph_{index + 1}"
            for index in range(len(graphs))
        ]

    if len(names) != len(graphs):
        raise ValueError(
            "The number of names must equal the number of graphs."
        )

    structural_values = structural_similarity_matrix(graphs)

    numeric_values = []
    final_values = []

    for row_index, graph_a in enumerate(graphs):
        numeric_row = []
        final_row = []

        for column_index, graph_b in enumerate(graphs):
            if comparison_mode == "equivalence":
                numeric_score = numeric_equivalence_score(
                    graph_a,
                    graph_b,
                )
            else:
                # graph_a is candidate; graph_b is requirement
                numeric_score = requirement_satisfaction_score(
                    candidate_graph=graph_a,
                    requirement_graph=graph_b,
                )

            structural_score = structural_values[
                row_index
            ][column_index]

            final_score = structural_score * numeric_score

            numeric_row.append(numeric_score)
            final_row.append(final_score)

        numeric_values.append(numeric_row)
        final_values.append(final_row)

    structural_df = pd.DataFrame(
        structural_values,
        index=names,
        columns=names,
    )

    numeric_df = pd.DataFrame(
        numeric_values,
        index=names,
        columns=names,
    )

    final_df = pd.DataFrame(
        final_values,
        index=names,
        columns=names,
    )

    return structural_df, numeric_df, final_df


# ============================================================
# Inspect a graph's numeric constraints
# ============================================================

def print_numeric_constraints(
    turtle_text: str,
    graph_name: str,
) -> None:
    graph = load_turtle(turtle_text)
    constraints = extract_numeric_constraints(graph)

    print(f"\nNumeric constraints for {graph_name}:")

    if not constraints:
        print("  None")
        return

    for constraint in constraints:
        print(
            f"  type={constraint.quantity_type}, "
            f"operator={constraint.operator}, "
            f"value={constraint.value}"
        )




In [32]:
from __future__ import annotations

import math
from typing import Any

import networkx as nx
from rdflib import Graph, Literal, OWL, RDF, RDFS


# ============================================================
# URI and node normalization
# ============================================================

def local_name(value: Any) -> str:
    """
    Convert an RDF URI to its local name and remove AMR2FRED
    generated instance numbers.

    Examples:
        fred:fiber_1              -> fiber
        fred:percentage-entity_2 -> percentage-entity
        pbrs:spin-01             -> spin-01
    """
    if isinstance(value, Literal):
        return str(value).strip().lower()

    text = str(value).rstrip("/")

    if "#" in text:
        name = text.rsplit("#", 1)[-1]
    else:
        name = text.rsplit("/", 1)[-1]

    parts = name.rsplit("_", 1)

    if len(parts) == 2 and parts[1].isdigit():
        name = parts[0]

    return name.lower().strip()


def is_schema_triple(subject, predicate, obj) -> bool:
    """
    Remove ontology declarations that do not describe the actual text.
    """
    if predicate in {
        RDFS.label,
        RDFS.subClassOf,
        RDFS.domain,
        RDFS.range,
    }:
        return True

    if predicate == RDF.type and obj in {
        OWL.ObjectProperty,
        OWL.DatatypeProperty,
        OWL.AnnotationProperty,
        OWL.Class,
    }:
        return True

    return False


def meaningful_types(rdf_graph: Graph, node) -> list[str]:
    """
    Get semantic RDF types for an instance.
    """
    result = []

    for node_type in rdf_graph.objects(node, RDF.type):
        if node_type in {
            OWL.ObjectProperty,
            OWL.DatatypeProperty,
            OWL.AnnotationProperty,
            OWL.Class,
        }:
            continue

        result.append(local_name(node_type))

    return sorted(set(result))


def make_node_label(rdf_graph: Graph, node) -> str:
    """
    Describe a node by semantic type rather than generated instance ID.
    """
    if isinstance(node, Literal):
        datatype = (
            local_name(node.datatype)
            if node.datatype
            else "literal"
        )

        return f"literal:{datatype}:{local_name(node)}"

    types = meaningful_types(rdf_graph, node)

    if types:
        return "|".join(types)

    return local_name(node)


# ============================================================
# Convert RDF into a directed NetworkX graph
# ============================================================

def rdf_to_networkx(turtle_text: str) -> nx.MultiDiGraph:
    rdf_graph = Graph()
    rdf_graph.parse(data=turtle_text, format="turtle")

    nx_graph = nx.MultiDiGraph()

    for subject, predicate, obj in rdf_graph:
        if is_schema_triple(subject, predicate, obj):
            continue

        subject_label = make_node_label(rdf_graph, subject)
        object_label = make_node_label(rdf_graph, obj)
        predicate_label = local_name(predicate)

        if subject not in nx_graph:
            nx_graph.add_node(
                subject,
                label=subject_label,
                literal=isinstance(subject, Literal),
            )

        if obj not in nx_graph:
            nx_graph.add_node(
                obj,
                label=object_label,
                literal=isinstance(obj, Literal),
            )

        nx_graph.add_edge(
            subject,
            obj,
            label=predicate_label,
        )

    return nx_graph


# ============================================================
# Node and edge matching
# ============================================================

def node_match(node_a: dict, node_b: dict) -> bool:
    """
    Nodes match only when their normalized semantic labels match.
    """
    return (
        node_a.get("label") == node_b.get("label")
        and node_a.get("literal") == node_b.get("literal")
    )


def edge_match(edge_a: dict, edge_b: dict) -> bool:
    """
    For MultiDiGraph, NetworkX may pass a mapping containing
    multiple parallel edges.
    """
    def extract_labels(edge_data: dict) -> set[str]:
        labels = set()

        if "label" in edge_data:
            labels.add(edge_data["label"])
            return labels

        for value in edge_data.values():
            if isinstance(value, dict) and "label" in value:
                labels.add(value["label"])

        return labels

    return extract_labels(edge_a) == extract_labels(edge_b)


# ============================================================
# Exact graph-edit similarity
# ============================================================

def graph_edit_similarity(
    turtle_a: str,
    turtle_b: str,
    timeout: float = 20.0,
) -> dict:
    """
    Compute structural similarity using graph edit distance.

    A score near 1 means similar node/edge structure.
    A score near 0 means dissimilar structure.

    Graph direction is preserved because MultiDiGraph is directed.
    """
    graph_a = rdf_to_networkx(turtle_a)
    graph_b = rdf_to_networkx(turtle_b)

    max_graph_size = (
        graph_a.number_of_nodes()
        + graph_a.number_of_edges()
        + graph_b.number_of_nodes()
        + graph_b.number_of_edges()
    )

    if max_graph_size == 0:
        return {
            "similarity": 1.0,
            "edit_distance": 0.0,
            "graph_a_nodes": 0,
            "graph_b_nodes": 0,
            "graph_a_edges": 0,
            "graph_b_edges": 0,
        }

    distance = nx.graph_edit_distance(
        graph_a,
        graph_b,
        node_match=node_match,
        edge_match=edge_match,
        timeout=timeout,
    )

    if distance is None:
        return {
            "similarity": None,
            "edit_distance": None,
            "error": "Graph edit calculation timed out",
            "graph_a_nodes": graph_a.number_of_nodes(),
            "graph_b_nodes": graph_b.number_of_nodes(),
            "graph_a_edges": graph_a.number_of_edges(),
            "graph_b_edges": graph_b.number_of_edges(),
        }

    similarity = 1.0 - min(float(distance) / max_graph_size, 1.0)

    return {
        "similarity": round(similarity, 4),
        "edit_distance": round(float(distance), 4),
        "graph_a_nodes": graph_a.number_of_nodes(),
        "graph_b_nodes": graph_b.number_of_nodes(),
        "graph_a_edges": graph_a.number_of_edges(),
        "graph_b_edges": graph_b.number_of_edges(),
    }


# ============================================================
# Fast path-based structural similarity
# ============================================================

def directed_path_features(
    graph: nx.MultiDiGraph,
    max_path_length: int = 3,
) -> set[str]:
    """
    Extract directed node-edge-node paths.

    Examples:
        fiber --hasQuality--> polyester
        fiber --quant--> percentage --hasDataValue--> 85

    This captures node flow without requiring exact whole-graph
    isomorphism.
    """
    features = set()

    # Length-one paths
    for source, target, edge_data in graph.edges(data=True):
        source_label = graph.nodes[source]["label"]
        target_label = graph.nodes[target]["label"]
        edge_label = edge_data["label"]

        features.add(
            f"{source_label}--{edge_label}-->{target_label}"
        )

    if max_path_length < 2:
        return features

    # Longer directed paths
    for start_node in graph.nodes:
        stack = [
            (
                start_node,
                [start_node],
                [],
            )
        ]

        while stack:
            current_node, nodes, edges = stack.pop()

            if len(edges) >= max_path_length:
                continue

            for _, next_node, edge_data in graph.out_edges(
                current_node,
                data=True,
            ):
                # Avoid cycles within a single extracted path.
                if next_node in nodes:
                    continue

                new_nodes = nodes + [next_node]
                new_edges = edges + [edge_data["label"]]

                parts = [
                    graph.nodes[new_nodes[0]]["label"]
                ]

                for edge_label, node in zip(
                    new_edges,
                    new_nodes[1:],
                ):
                    parts.append(f"--{edge_label}-->")
                    parts.append(graph.nodes[node]["label"])

                features.add("".join(parts))

                stack.append(
                    (
                        next_node,
                        new_nodes,
                        new_edges,
                    )
                )

    return features


def path_similarity(
    turtle_a: str,
    turtle_b: str,
    max_path_length: int = 3,
) -> dict:
    """
    Jaccard similarity over directed semantic paths.
    """
    graph_a = rdf_to_networkx(turtle_a)
    graph_b = rdf_to_networkx(turtle_b)

    paths_a = directed_path_features(
        graph_a,
        max_path_length=max_path_length,
    )
    paths_b = directed_path_features(
        graph_b,
        max_path_length=max_path_length,
    )

    union = paths_a | paths_b
    intersection = paths_a & paths_b

    similarity = (
        len(intersection) / len(union)
        if union
        else 1.0
    )

    return {
        "similarity": round(similarity, 4),
        "shared_paths": len(intersection),
        "graph_a_paths": len(paths_a),
        "graph_b_paths": len(paths_b),
        "matching_paths": sorted(intersection),
    }

def containment_path_similarity(
    candidate_turtle: str,
    reference_turtle: str,
    max_path_length: int = 3,
) -> dict:
    """
    Measures how much of the candidate graph's structure appears
    in the reference graph.

    This is asymmetric:

        candidate Graph 1 -> reference Graph 2

    is not necessarily the same as:

        candidate Graph 2 -> reference Graph 1
    """

    candidate_graph = rdf_to_networkx(candidate_turtle)
    reference_graph = rdf_to_networkx(reference_turtle)

    candidate_paths = directed_path_features(
        candidate_graph,
        max_path_length=max_path_length,
    )

    reference_paths = directed_path_features(
        reference_graph,
        max_path_length=max_path_length,
    )

    shared_paths = candidate_paths & reference_paths

    # Important change:
    # divide by candidate paths, not the union
    score = (
        len(shared_paths) / len(candidate_paths)
        if candidate_paths
        else 0.0
    )

    return {
        "similarity": round(score, 4),
        "shared_paths": len(shared_paths),
        "candidate_paths": len(candidate_paths),
        "reference_paths": len(reference_paths),
        "matching_paths": sorted(shared_paths),
    }
# ============================================================
# Compare one graph against several graphs
# ============================================================

def compare_against_reference(
    reference_turtle: str,
    candidates: list[tuple[str, str]],
    method: str = "path",
    threshold: float = 0.20,
) -> list[dict]:
    """
    candidates:
        [
            ("graph_2", turtle_graph_2),
            ("graph_3", turtle_graph_3),
        ]
    """
    results = []

    for candidate_name, candidate_turtle in candidates:
        if method == "edit":
            result = graph_edit_similarity(
                reference_turtle,
                candidate_turtle,
            )
        elif method == "path":
            result = path_similarity(
                reference_turtle,
                candidate_turtle,
            )
        else:
            raise ValueError(
                "method must be 'path' or 'edit'"
            )

        score = result.get("similarity")

        compatible = (
            score is not None
            and score >= threshold
        )

        result["candidate"] = candidate_name
        result["compatible"] = compatible
        results.append(result)

    results.sort(
        key=lambda item: (
            item["similarity"]
            if item["similarity"] is not None
            else -1
        ),
        reverse=True,
    )

    return results


def print_comparison_results(results: list[dict]) -> None:
    for result in results:
        score = result.get("similarity")

        score_text = (
            f"{score:.4f}"
            if score is not None
            else "timeout"
        )

        print(
            f"{result['candidate']}: "
            f"{'YES' if result['compatible'] else 'NO'} "
            f"(structural similarity={score_text})"
        )




In [33]:
result_1 = containment_path_similarity(
    candidate_turtle=turtle_graph_1,
    reference_turtle=turtle_graph_2,
)

result_3 = containment_path_similarity(
    candidate_turtle=turtle_graph_3,
    reference_turtle=turtle_graph_2,
)

print(
    f"graph_1 similarity: {result_1['similarity']:.4f} "
    f"({result_1['shared_paths']}/"
    f"{result_1['candidate_paths']} candidate paths matched)"
)

print(
    f"graph_3 similarity: {result_3['similarity']:.4f} "
    f"({result_3['shared_paths']}/"
    f"{result_3['candidate_paths']} candidate paths matched)"
)

graph_1 similarity: 0.0656 (4/61 candidate paths matched)
graph_3 similarity: 0.0610 (5/82 candidate paths matched)


In [30]:
text="fibres, synthetic staple fibres, of polyesters, not carded, combed or otherwise processed for spinning."

turtle_graph_1 = converter.translate(
    text=text,
    serialize=True,
    mode=Glossary.RdflibMode.TURTLE,
    post_processing=False,
)
text = "Polyester staple fiber of Okolona Mississippi. The submitted staple fiber that you indictae is to be used for stuffing furniture such as sofas and chairs. You state that the fiber has no sheath and the denier measure is 15 (165 decitex)"

turtle_graph_2 = converter.translate(
    text=text,
    serialize=True,
    mode=Glossary.RdflibMode.TURTLE,
    post_processing=False,
)

text="yarn not sewing thread single of synthetic staple fibres, containing 85% or more by weight of polyester, not put up for retail sale"
turtle_graph_3 = converter.translate(
    text=text,
    serialize=True,
    mode=Glossary.RdflibMode.TURTLE,
    post_processing=False,
)

In [31]:


results = compare_against_reference(
    reference_turtle=turtle_graph_2,
    candidates=[
        ("graph_1", turtle_graph_1),
        ("graph_3", turtle_graph_3),
    ],
    method="path",
    threshold=0.20,
)

print_comparison_results(results)

graph_3: NO (structural similarity=0.0208)
graph_1: NO (structural similarity=0.0182)


In [23]:

graph_texts = [
turtle_graph_1,
turtle_graph_2,
turtle_graph_3,
]

graph_names = [
"graph_8_percent",
"graph_at_least_5_percent",
"graph_4_percent",
]
def is_compatible(
    candidate_turtle: str,
    requirement_turtle: str,
) -> bool:
    candidate_graph = load_turtle(candidate_turtle)
    requirement_graph = load_turtle(requirement_turtle)

    score = requirement_satisfaction_score(
        candidate_graph=candidate_graph,
        requirement_graph=requirement_graph,
        context_threshold=0.5,
    )

    return score == 1.0
print(
    "Graph 1 compatible with Graph 2:",
    "YES" if is_compatible(
        turtle_graph_1,
        turtle_graph_2,
    ) else "NO",
)

print(
    "Graph 3 compatible with Graph 2:",
    "YES" if is_compatible(
        turtle_graph_3,
        turtle_graph_2,
    ) else "NO",
)

Graph 1 compatible with Graph 2: NO
Graph 3 compatible with Graph 2: NO


In [5]:
from py_amr2fred import Amr2fred, Glossary

converter = Amr2fred()

text = "Narrow Woven Fabrics 8% Elastomeric Yarn"

graph_text = converter.translate(
    text=text,
    serialize=True,
    mode=Glossary.RdflibMode.TURTLE,
    post_processing=False,
)

print(graph_text)
print("=====================")

text = "Narrow Woven Fabrics, Other Than Goods Of Heading 5807; Others Woven Fabrics, Containing By Weight 5% Or More Of Elastomeric Yarn Or Rubber Thread"

graph_text2 = converter.translate(
    text=text,
    serialize=True,
    mode=Glossary.RdflibMode.TURTLE,
    post_processing=False,
)

print(graph_text2)

@prefix amr: <https://w3id.org/framester/amr/> .
@prefix boxing: <http://www.ontologydesignpatterns.org/ont/boxer/boxing.owl#> .
@prefix dul: <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#> .
@prefix fred: <http://www.ontologydesignpatterns.org/ont/fred/domain.owl#> .
@prefix fschema: <https://w3id.org/framester/schema/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix pblr: <https://w3id.org/framester/data/propbank-3.4.0/LocalRole/> .
@prefix pbrs: <https://w3id.org/framester/data/propbank-3.4.0/RoleSet/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix va: <http://verbatlas.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

dul:hasDataValue a owl:DatatypeProperty .

dul:hasMember a owl:ObjectProperty .

dul:hasQuality a owl:ObjectProperty .

dul:precedes a owl:ObjectProperty .

fred:narrow_1 a pbrs:narrow-02 ;
    dul:precedes fred:conjunct_1 ;
    pblr:narrow-02.thing-that-is-narrow fred:weave_1 .

amr:quant a owl:ObjectProperty .

pblr:na

INFO:py_amr2fred.taf_post_processor:Downloading index_enwiki-latest db...
MBytes: 100%|██████████| 832/832 [00:36<00:00, 23.10it/s]
INFO:py_amr2fred.taf_post_processor:Extracting db from zip-file...
INFO:py_amr2fred.taf_post_processor:Returning initial graph, no entities to be linked to wikidata
